In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
    substring,
    trim,
    when,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_case_code_qc(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args):
        date_filter = f"businessdate = {run_date!r}"

        # -------SCHEMA
        #schema = (
        #    "CaseCodeId	string,"
        #    "SourceSystem	string,"
        #    "PortfolioKey	string,"
        #    "PortfolioName	string,"
        #    "IngestionTimestamp	timestamp,"
        #    "BatchId	string,"
        #    "CaseCodeKey	int,"
        #    "CaseCodePrimary	int,"
        #    "CaseRelationTypeCode	int,"
        #    "CustCode	int,"
        #    "DebtCode	int,"
        #    "QueueCode	int,"
        #    "Queue	string,"
        #    "AdminStatusCode	int,"
        #    "AdminStatus	string,"
        #    "AdminStatusDate	date,"
        #    "AdminStatusExpirationDate	date,"
        #    "StrategyStepCode	int,"
        #    "StrategyStep	string,"
        #    "CaseSourceCode	string,"
        #    "CurrentlyOpenFlag	int,"
        #    "LPS	int,"
        #    "LPSStatus	string,"
        #    "LPSDate	timestamp,"
        #    "ParentStrategyStepCode	int,"
        #    "ParentStrategyStep	string,"
        #    "StrategyTypeCode	string,"
        #    "StrategyType	string,"
        #    "StrategyFlowCode	string,"
        #    "StrategyFlow	string,"
        #    "WatchlistFlag	string,"
        #    "SubsidyEndDate	timestamp,"
        #    "LegalFileLocation	string,"
        #    "ApplicationDecisionEverDate	timestamp,"
        #    "GovernmentSubsidyType	string,"
        #    "AgencyCode	int,"
        #    "Agency	string,"
        #    "PacketCode	int,"
        #    "Packet	string,"
        #    "AppealFlag	int,"
        #    "ConfiscationCurrentAgencyCode	int,"
        #    "ConfiscationCurrentAgency	string,"
        #    "CurrentStageConfiscationCode	int,"
        #    "CurrentStageConfiscation	string,"
        #    "ForcedPrenotationCurrentAgencyCode	int,"
        #    "ForcedPrenotationCurrentAgency	string,"
        #    "CurrentStageForcedPrenotationCode	int,"
        #    "CurrentStageForcedPrenotation	string,"
        #    "RegularSuitCurrentAgencyCode	int,"
        #    "RegularSuitCurrentAgency	string,"
        #    "CurrentStageRegularSuitCode	int,"
        #    "CurrentStageRegularSuit	string,"
        #    "PaymentOrderCurrentAgencyCode	int,"
        #    "PaymentOrderCurrentAgency	string,"
        #    "CurrentStagePaymentOrderCode	int,"
        #    "CurrentStagePaymentOrder	string,"
        #    "TransactionPerimeterCode	string,"
        #    "AccountNumber string,"
        #    "MinimumPaymentAmount decimal(19,4),"
        #    "AppealCode string,"
        #    "ActiveAppealFlag boolean,"
        #    "DecisionDate date,"
        #    "CommitteeDecisionTypeCode string,"
        #    "CommitteeDecisionType string,"
        #    "CaseCodeKeyOld int", #new
        #    "BusinessDate	date"
        #)

        cluster_columns = ["BusinessDate"]

        # -------CHOOSE PATH

        curated_table_path = f"{curated_zone}.CaseCode"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "qc",
            )
        else:
            logging.info(
                "Tables does not exist, no need to delete entries for business date!"
            )

        # --------APPLY FILTERS AND SELECT COLUMNS FROM THE TABLES WE NEED

        # ================== QC_case_master ==================

        case_master_extra_filters = None

        if case_master_extra_filters is not None:
            case_master_extra_filters.append(date_filter)
            case_master_final_filters = " AND ".join(case_master_extra_filters)
        else:
            case_master_final_filters = date_filter

        case_master_selected_fields = [
            "case_code",
            "VEND_CODE",
            "case_code_Primary",
            "case_relation_type_code",
            "cust_code",
            "debt_code",
            "que_code",
            "que_desc",
            "caco_administration_status_code",
            "CACO_ADMINISTRATION_STATUS_DATE",
            "CACO_ADMINISTRATION_STATUS_EXPIRATION_DATE",
            "strategy_step_id",
            "case_currently_open_flag",
            "LPS",
            "LPS_Status",
            "LPS_date",
            "CEXT_DEDICATED_PROPERTY_02_NVARCHAR",
            "CEXT_GENERIC_DATETIME_11",
            "old_case_code",
            "CAEX_CUSTOM_TEXT_49",
            "BusinessDate",
        ]

        case_master = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qc_case_master",
            case_master_final_filters,
            case_master_selected_fields,
        )
        # ================== portfolio ==================

        portfolio_extra_filters = None

        if portfolio_extra_filters is not None:
            portfolio_extra_filters.append(date_filter)
            portfolio_final_filters = " AND ".join(portfolio_extra_filters)
        else:
            portfolio_final_filters = date_filter  # possible need change date

        portfolio_selected_fields = [
            "PortfolioName",
            "vendor_code_qualco",
        ]

        portfolio = load_landed_portfolio_table(  # noqa: F841
            spark,
            curated_zone,  # type: ignore
            "portfolio",
            portfolio_final_filters,
            portfolio_selected_fields,
        )
        # ================== QC_LST_0F_VAL ==================

        lst_of_val_extra_filters = None

        if lst_of_val_extra_filters is not None:
            lst_of_val_extra_filters.append(date_filter)
            lst_of_val_final_filters = " AND ".join(lst_of_val_extra_filters)
        else:
            lst_of_val_final_filters = date_filter

        lst_of_val_selected_fields = [
            "lov_desc",
            "lov_code",
            "lovt_desc",
        ]

        lst_of_val = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qc_lst_of_val",
            lst_of_val_final_filters,
            lst_of_val_selected_fields,
        )
        # ================== QC_strategy_step ==================

        strategy_step_extra_filters = None

        if strategy_step_extra_filters is not None:
            strategy_step_extra_filters.append(date_filter)
            strategy_step_final_filters = " AND ".join(strategy_step_extra_filters)
        else:
            strategy_step_final_filters = date_filter

        strategy_step_selected_fields = [
            "strategy_step_desc",
            "strategy_step_id",
            "strategy_id",
            "strategy_desc",
            "strategy_area_id",
            "strategy_area_desc",
        ]

        strategy_step = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qc_strategy_step",
            strategy_step_final_filters,
            strategy_step_selected_fields,
        )
        # ================== qc_customer ==================

        qc_customer_extra_filters = None

        if qc_customer_extra_filters is not None:
            qc_customer_extra_filters.append(date_filter)
            qc_customer_final_filters = " AND ".join(qc_customer_extra_filters)
        else:
            qc_customer_final_filters = date_filter

        qc_customer_selected_fields = [
            "cust_code",
            "cdi",
        ]

        qc_customer = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qc_customer",
            qc_customer_final_filters,
            qc_customer_selected_fields,
        )
        # ================== profits_customer ==================

        profits_customer_extra_filters = None

        if profits_customer_extra_filters is not None:
            profits_customer_extra_filters.append(date_filter)
            profits_customer_final_filters = " AND ".join(
                profits_customer_extra_filters
            )
        else:
            profits_customer_final_filters = date_filter

        profits_customer_selected_fields = [
            "CDI",
        ]

        profits_customer = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "profits_customer",
            profits_customer_final_filters,
            profits_customer_selected_fields,
        )
        # ================== profits_customer_RF_CODES ==================

        profits_customer_rf_codes_extra_filters = None

        if profits_customer_rf_codes_extra_filters is not None:
            profits_customer_rf_codes_extra_filters.append(date_filter)
            profits_customer_rf_codes_final_filters = " AND ".join(
                profits_customer_rf_codes_extra_filters
            )
        else:
            profits_customer_rf_codes_final_filters = date_filter

        profits_customer_rf_codes_selected_fields = [
            "CUSTOMER_ID_NO",
            "account_number",
        ]

        profits_customer_RF_CODES = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "profits_customer_rf_codes",
            profits_customer_rf_codes_final_filters,
            profits_customer_rf_codes_selected_fields,
        )

        # -------SET THE CONDITIONS FOR JOINS
        case_master_Portfolio_join_condition = (
            case_master["VEND_CODE"] == portfolio["VENDOR_CODE_QUALCO"]
        )

        case_master_qc_customer_join_condition = (
            case_master["CUST_CODE"] == qc_customer["CUST_CODE"]
        )

        profits_customer_qc_customer_join_condition = (
            profits_customer["CDI"] == qc_customer["CDI"]
        )

        profits_customer_profits_customer_Rf_Codes_join_condition = (
            trim(profits_customer["CDI"])
            == trim(profits_customer_RF_CODES["CUSTOMER_ID_NO"])
        ) & (
            profits_customer_RF_CODES["ACCOUNT_NUMBER"] == case_master["OLD_CASE_CODE"]
        )
        case_master_strategy_step_join_condition = (
            case_master["strategy_step_ID"] == strategy_step["strategy_step_ID"]
        )
        case_master_Lst_of_lov_join_condition = (
            case_master["caco_administration_status_code"] == lst_of_val["lov_code"]
        ) & (lst_of_val["lovt_desc"] == "ADMIN_STATUS")

        # -------APPLY THE JOINS
        case_code_df = (
            case_master.join(
                portfolio,
                case_master_Portfolio_join_condition,
                "inner",
            )
            .join(
                qc_customer,
                case_master_qc_customer_join_condition,
                "inner",
            )
            .join(
                profits_customer,
                profits_customer_qc_customer_join_condition,
                "inner",
            )
            .join(
                profits_customer_RF_CODES,
                profits_customer_profits_customer_Rf_Codes_join_condition,
                "left",
            )
            .join(
                strategy_step,
                case_master_strategy_step_join_condition,
                "left",
            )
            .join(
                lst_of_val,
                case_master_Lst_of_lov_join_condition,
                "left",
            )
        )

        # --------SET THE KEYS FOR HASHKEY BASED ON DATA MAPPING
        case_code_df_party_CaseCodeId_primary_key_fields = [
            case_master.case_code,
            lit("qc"),
        ]

        # -------CREATION OF HΑSKEY
        case_code_df = case_code_df.withColumn(
            "CaseCodeId",
            sha2(
                concat_ws(";", *case_code_df_party_CaseCodeId_primary_key_fields),
                256,
            ),
        ).withColumn(
            "DecisionDate",
            when(
                case_master["CEXT_GENERIC_DATETIME_11"].isNull(),
                lit("1900-01-01").cast("date"),
            ).otherwise(case_master["CEXT_GENERIC_DATETIME_11"].cast("date")),
        )

        # -------SELECTION OF FINAL COLUMNS
        result_df_qc = case_code_df.select(
            case_code_df.CaseCodeId.alias("CaseCodeId"),
            lit("qc").alias("SourceSystem"),
            case_master.VEND_CODE.cast("string").alias("PortfolioKey"),
            portfolio.PortfolioName.cast("string").alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens"))
            .cast("timestamp")
            .alias("IngestionTimestamp"),
            lit(None).alias("BatchId"),
            case_master.case_code.cast("int").alias("CaseCodeKey"),
            case_master.case_code_Primary.cast("int").alias("CaseCodePrimary"),
            case_master.case_relation_type_code.cast("int").alias(
                "CaseRelationTypeCode"
            ),
            case_master.cust_code.cast("int").alias("CustCode"),
            case_master.debt_code.cast("int").alias("DebtCode"),
            case_master.que_code.cast("int").alias("QueueCode"),
            case_master.que_desc.cast("string").alias("Queue"),
            case_master.caco_administration_status_code.cast("int").alias(
                "AdminStatusCode"
            ),
            lst_of_val.lov_desc.cast("string").alias("AdminStatus"),
            case_master.CACO_ADMINISTRATION_STATUS_DATE.cast("date").alias(
                "AdminStatusDate"
            ),
            case_master.CACO_ADMINISTRATION_STATUS_EXPIRATION_DATE.cast("date").alias(
                "AdminStatusExpirationDate"
            ),
            case_master.strategy_step_id.cast("int").alias("StrategyStepCode"),
            strategy_step.strategy_step_desc.cast("string").alias("StrategyStep"),
            lit(None).alias("CaseSourceCode"),
            case_master.case_currently_open_flag.cast("int").alias("CurrentlyOpenFlag"),
            case_master.LPS.cast("int").alias("LPS"),
            case_master.LPS_Status.cast("string").alias("LPSStatus"),
            case_master.LPS_date.cast("timestamp").alias("LPSDate"),
            strategy_step.strategy_id.cast("int").alias("ParentStrategyStepCode"),
            strategy_step.strategy_desc.cast("string").alias("ParentStrategyStep"),
            strategy_step.strategy_area_id.cast("string").alias("StrategyTypeCode"),
            strategy_step.strategy_area_desc.cast("string").alias("StrategyType"),
            lit(None).alias("StrategyFlowCode"),
            lit(None).alias("StrategyFlow"),
            lit(None).alias("WatchlistFlag"),
            lit(None).alias("SubsidyEndDate"),
            lit(None).alias("LegalFileLocation"),
            lit(None).alias("ApplicationDecisionEverDate"),
            lit(None).alias("GovernmentSubsidyType"),
            lit(None).alias("AgencyCode"),
            lit(None).alias("Agency"),
            lit(None).alias("PacketCode"),
            lit(None).alias("Packet"),
            lit(None).alias("AppealFlag"),
            lit(None).alias("ConfiscationCurrentAgencyCode"),
            lit(None).alias("ConfiscationCurrentAgency"),
            lit(None).alias("CurrentStageConfiscationCode"),
            lit(None).alias("CurrentStageConfiscation"),
            lit(None).alias("ForcedPrenotationCurrentAgencyCode"),
            lit(None).alias("ForcedPrenotationCurrentAgency"),
            lit(None).alias("CurrentStageForcedPrenotationCode"),
            lit(None).alias("CurrentStageForcedPrenotation"),
            lit(None).alias("RegularSuitCurrentAgencyCode"),
            lit(None).alias("RegularSuitCurrentAgency"),
            lit(None).alias("CurrentStageRegularSuitCode"),
            lit(None).alias("CurrentStageRegularSuit"),
            lit(None).alias("PaymentOrderCurrentAgencyCode"),
            lit(None).alias("PaymentOrderCurrentAgency"),
            lit(None).alias("CurrentStagePaymentOrderCode"),
            lit(None).alias("CurrentStagePaymentOrder"),
            case_master.CEXT_DEDICATED_PROPERTY_02_NVARCHAR.cast("string").alias(
                "TransactionPerimeterCode"
            ),
            lit(None).alias("AccountNumber"),
            lit(None).alias("MinimumPaymentAmount"),
            lit(-1).cast("int").alias("AppealCode"),
            lit(None).cast("boolean").alias("ActiveAppealFlag"),
            case_code_df.DecisionDate.alias("DecisionDate"),
            case_code_df.CAEX_CUSTOM_TEXT_49.alias("CommitteeDecisionTypeCode"),
            case_code_df.CAEX_CUSTOM_TEXT_49.alias("CommitteeDecisionType"),
            lit(None).cast("int").alias("CaseCodeKeyOld"), ## new
            case_master.BusinessDate.cast("date").alias("BusinessDate"),
        )
        # ----------------------------------------------APPEND DATA TO CURATED-------------------------------------------------------------------------
        result_df_qc = result_df_qc.dropDuplicates()
        result_df_qc.write.format("delta").clusterBy("BusinessDate").mode("append").saveAsTable(curated_table_path)

    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")

In [0]:
load_curated_case_code_qc(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)